In [54]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score, 
    precision_recall_fscore_support, 
    mean_squared_error,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
import logging
from pathlib import Path
from typing import Dict, Tuple, Any, List
import matplotlib.pyplot as plt
import seaborn as sns

In [55]:
%run 02_data_processor.ipynb

2024-10-29 18:43:59,744 - INFO - 正在讀取數據文件: D:\Min\Python\Project\FA_Data\meta_data\all_stocks_data.csv



正在載入台積電 (2330) 的數據...


2024-10-29 18:44:09,240 - INFO - 原始數據形狀: (2450900, 29)
2024-10-29 18:44:09,431 - INFO - 過濾後的數據形狀 (股票 2330): (2584, 29)
2024-10-29 18:44:09,435 - INFO - 開始日期過濾後的數據形狀: (437, 29)
2024-10-29 18:44:09,436 - INFO - 結束日期過濾後的數據形狀: (239, 29)
2024-10-29 18:44:09,456 - INFO - 正在讀取數據文件: D:\Min\Python\Project\FA_Data\meta_data\all_stocks_data.csv



=== 股票數據摘要 ===
股票代碼: 2330
股票名稱: 台積電
數據期間: 2023-01-03 到 2023-12-29
交易天數: 239 天

--- 價格統計 ---
平均收盤價: 543.45
最高價: 594.00
最低價: 443.00
價格波動範圍: 151.00
標準差: 29.62

--- 交易統計 ---
平均日成交量: 27,535,566.40
平均日成交筆數: 29,635.65
平均日成交金額: 14,980,440,520.00

--- 技術指標概覽 ---

TREND類指標:
SMA30        - 均值: 536.47, 標準差: 28.85
DEMA30       - 均值: 543.00, 標準差: 29.66
EMA30        - 均值: 536.41, 標準差: 28.16
TSF          - 均值: 543.78, 標準差: 31.83
middleband   - 均值: 566.27, 標準差: 26.47

MOMENTUM類指標:
RSI          - 均值: 53.59, 標準差: 10.37
MACD         - 均值: 3.21, 標準差: 7.07
MACD_signal  - 均值: 3.12, 標準差: 6.46
MACD_hist    - 均值: 0.09, 標準差: 2.59

VOLATILITY類指標:
slowk        - 均值: 52.71, 標準差: 28.52
slowd        - 均值: 52.39, 標準差: 25.99
SAR          - 均值: 539.43, 標準差: 34.31

=== 數據載入完成 ===

正在處理 2330 的特徵...


2024-10-29 18:44:18,845 - INFO - 原始數據形狀: (2450900, 29)
2024-10-29 18:44:19,047 - INFO - 過濾後的數據形狀 (股票 2330): (2584, 29)
2024-10-29 18:44:19,053 - INFO - 開始日期過濾後的數據形狀: (437, 29)
2024-10-29 18:44:19,055 - INFO - 結束日期過濾後的數據形狀: (239, 29)



=== 股票數據摘要 ===
股票代碼: 2330
股票名稱: 台積電
數據期間: 2023-01-03 到 2023-12-29
交易天數: 239 天

--- 價格統計 ---
平均收盤價: 543.45
最高價: 594.00
最低價: 443.00
價格波動範圍: 151.00
標準差: 29.62

--- 交易統計 ---
平均日成交量: 27,535,566.40
平均日成交筆數: 29,635.65
平均日成交金額: 14,980,440,520.00

--- 技術指標概覽 ---

TREND類指標:
SMA30        - 均值: 536.47, 標準差: 28.85
DEMA30       - 均值: 543.00, 標準差: 29.66
EMA30        - 均值: 536.41, 標準差: 28.16
TSF          - 均值: 543.78, 標準差: 31.83
middleband   - 均值: 566.27, 標準差: 26.47

MOMENTUM類指標:
RSI          - 均值: 53.59, 標準差: 10.37
MACD         - 均值: 3.21, 標準差: 7.07
MACD_signal  - 均值: 3.12, 標準差: 6.46
MACD_hist    - 均值: 0.09, 標準差: 2.59

VOLATILITY類指標:
slowk        - 均值: 52.71, 標準差: 28.52
slowd        - 均值: 52.39, 標準差: 25.99
SAR          - 均值: 539.43, 標準差: 34.31

=== 特徵生成完成 ===
特徵數量: 13
特徵名稱: ['趨勢強度', '均線糾結度', '波動率', '振幅', '量能趨勢', '量比', 'RSI_動能', 'MACD_動能', 'RSI', 'MACD', 'KD_差值', '本益比_相對值', '技術綜合評分']

特徵統計:
               趨勢強度         均線糾結度           波動率            振幅          量能趨勢  \
count  2.390000e+02  2.390000e+0

In [56]:
class ModelAnalyzer:
    """模型分析和視覺化工具"""
    
    def __init__(self, figure_size: Tuple[int, int] = (15, 10)):
        self.figure_size = figure_size
        # 設置 matplotlib 樣式
        plt.style.use('default')  # 使用默認樣式
        # 設置 seaborn 樣式
        sns.set_theme(style="whitegrid")  # 使用 seaborn 的網格樣式
    
    def plot_cumulative_returns(self, returns: pd.Series, 
                              benchmark_returns: pd.Series = None,
                              title: str = "策略累積報酬") -> None:
        """繪製累積報酬曲線"""
        plt.figure(figsize=self.figure_size)
        
        # 計算累積報酬
        cum_returns = (1 + returns).cumprod()
        plt.plot(cum_returns.index, cum_returns.values, 
                label='策略報酬', linewidth=2)
        
        if benchmark_returns is not None:
            cum_benchmark = (1 + benchmark_returns).cumprod()
            plt.plot(cum_benchmark.index, cum_benchmark.values, 
                    label='大盤報酬', linewidth=2, linestyle='--')
        
        plt.title(title, fontsize=15)
        plt.xlabel('日期', fontsize=12)
        plt.ylabel('累積報酬', fontsize=12)
        plt.grid(True)
        plt.legend(fontsize=12)
        plt.tight_layout()
        plt.show()
    
    def plot_confusion_matrix(self, y_true: np.ndarray, 
                            y_pred: np.ndarray,
                            title: str = "混淆矩陣") -> None:
        """繪製混淆矩陣熱圖"""
        plt.figure(figsize=(10, 8))
        cm = confusion_matrix(y_true, y_pred)
        
        # 使用seaborn繪製熱圖
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['下跌', '上漲'],
                   yticklabels=['下跌', '上漲'])
        
        plt.title(title, fontsize=15)
        plt.xlabel('預測', fontsize=12)
        plt.ylabel('實際', fontsize=12)
        plt.tight_layout()
        plt.show()
    
    def plot_roc_curve(self, y_true: np.ndarray, 
                      y_prob: np.ndarray,
                      title: str = "ROC曲線") -> None:
        """繪製ROC曲線"""
        plt.figure(figsize=(10, 8))
        
        fpr, tpr, _ = roc_curve(y_true, y_prob[:, 1])
        auc = roc_auc_score(y_true, y_prob[:, 1])
        
        plt.plot(fpr, tpr, label=f'ROC曲線 (AUC = {auc:.3f})',
                linewidth=2)
        plt.plot([0, 1], [0, 1], 'k--', label='隨機猜測')
        
        plt.title(title, fontsize=15)
        plt.xlabel('假陽性率 (FPR)', fontsize=12)
        plt.ylabel('真陽性率 (TPR)', fontsize=12)
        plt.grid(True)
        plt.legend(fontsize=12)
        plt.tight_layout()
        plt.show()
    
    def plot_feature_importance(self, feature_importance: Dict[str, float],
                              title: str = "特徵重要性排序") -> None:
        """繪製特徵重要性條形圖"""
        plt.figure(figsize=self.figure_size)
        
        # 排序特徵重要性
        importance_df = pd.DataFrame({
            'feature': list(feature_importance.keys()),
            'importance': list(feature_importance.values())
        }).sort_values('importance', ascending=True)
        
        # 使用seaborn繪製條形圖
        sns.barplot(data=importance_df, x='importance', y='feature')
        plt.title(title, fontsize=15)
        plt.xlabel('重要性', fontsize=12)
        plt.ylabel('特徵', fontsize=12)
        plt.tight_layout()
        plt.show()
    
    def plot_trading_signals(self, prices: pd.Series, 
                           signals: pd.Series,
                           title: str = "交易信號視覺化") -> None:
        """繪製交易信號"""
        # 創建Plotly圖形
        fig = go.Figure()
        
        # 添加價格線
        fig.add_trace(go.Scatter(
            x=prices.index,
            y=prices.values,
            mode='lines',
            name='價格',
            line=dict(color='blue')
        ))
        
        # 添加買入信號
        buy_signals = signals[signals == 1]
        if not buy_signals.empty:
            fig.add_trace(go.Scatter(
                x=buy_signals.index,
                y=prices[buy_signals.index],
                mode='markers',
                name='買入信號',
                marker=dict(color='red', size=10, symbol='triangle-up')
            ))
        
        # 添加賣出信號
        sell_signals = signals[signals == -1]
        if not sell_signals.empty:
            fig.add_trace(go.Scatter(
                x=sell_signals.index,
                y=prices[sell_signals.index],
                mode='markers',
                name='賣出信號',
                marker=dict(color='green', size=10, symbol='triangle-down')
            ))
        
        # 更新布局
        fig.update_layout(
            title=title,
            xaxis_title='日期',
            yaxis_title='價格',
            template='plotly_white'
        )
        
        fig.show()

In [57]:
class ModelOptimizer:
    """模型優化工具"""
    
    def __init__(self, model, param_grid: Dict[str, List[Any]]):
        """
        初始化模型優化器
        
        Args:
            model: 要優化的模型
            param_grid: 參數網格，格式為 {參數名: [可能的值]}
        """
        self.model = model
        self.param_grid = param_grid
        self.best_params = None
        self.cv_results = None
    
    def optimize(self, X: np.ndarray, y: np.ndarray, 
                cv: int = 5, scoring: str = 'accuracy') -> Dict[str, Any]:
        """
        執行網格搜索優化
        
        Args:
            X: 特徵矩陣
            y: 標籤向量
            cv: 交叉驗證折數
            scoring: 評分標準
            
        Returns:
            優化結果字典
        """
        try:
            logging.info("開始網格搜索...")
            
            # 創建時間序列分割器
            tscv = TimeSeriesSplit(n_splits=cv)
            
            # 執行網格搜索
            grid_search = GridSearchCV(
                estimator=self.model,
                param_grid=self.param_grid,
                cv=tscv,
                scoring=scoring,
                n_jobs=-1,  # 使用所有CPU核心
                verbose=1,
                return_train_score=True
            )
            
            # 擬合數據
            grid_search.fit(X, y)
            
            # 保存結果
            self.best_params = grid_search.best_params_
            self.cv_results = grid_search.cv_results_
            
            logging.info(f"最佳參數: {self.best_params}")
            logging.info(f"最佳分數: {grid_search.best_score_:.4f}")
            
            return {
                'best_params': self.best_params,
                'best_score': grid_search.best_score_,
                'cv_results': self.cv_results
            }
            
        except Exception as e:
            logging.error(f"優化過程中出錯: {str(e)}")
            raise
    
    def plot_optimization_results(self) -> None:
        """繪製優化結果分析圖"""
        if self.cv_results is None:
            logging.warning("沒有可用的優化結果可供繪圖")
            return
            
        try:
            # 創建結果DataFrame
            results_df = pd.DataFrame(self.cv_results)
            
            # 設置圖形風格
            plt.style.use('default')
            
            # 創建子圖
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
            
            # 繪製驗證分數分布
            mean_scores = results_df['mean_test_score']
            std_scores = results_df['std_test_score']
            
            # 繪製平均分數
            ax1.plot(range(len(mean_scores)), mean_scores, 'o-', label='平均分數')
            
            # 添加標準差區間
            ax1.fill_between(
                range(len(mean_scores)),
                mean_scores - std_scores,
                mean_scores + std_scores,
                alpha=0.2
            )
            
            ax1.set_title('交叉驗證分數分布')
            ax1.set_xlabel('參數組合索引')
            ax1.set_ylabel('準確率')
            ax1.grid(True)
            
            # 繪製最佳分數隨參數變化的趨勢
            best_score_idx = results_df['rank_test_score'] == 1
            best_params = results_df[best_score_idx].iloc[0]
            
            # 找出變化的參數
            varying_params = [param for param in self.param_grid.keys() 
                            if len(set(results_df[f'param_{param}'])) > 1]
            
            if varying_params:
                param = varying_params[0]  # 選擇第一個變化的參數進行展示
                param_values = results_df[f'param_{param}']
                scores = results_df['mean_test_score']
                
                ax2.plot(param_values, scores, 'o-')
                ax2.set_title(f'分數隨 {param} 的變化')
                ax2.set_xlabel(param)
                ax2.set_ylabel('準確率')
                ax2.grid(True)
            
            plt.tight_layout()
            plt.show()
            
            # 打印最佳參數組合
            logging.info("\n最佳參數組合:")
            for param in self.param_grid.keys():
                value = best_params[f'param_{param}']
                logging.info(f"{param}: {value}")
            
        except Exception as e:
            logging.error(f"繪製優化結果時出錯: {str(e)}")
            raise

In [58]:
class ModelConfig:
    """模型配置"""
    # 基本參數
    TREND_MODEL_PARAMS = {
        'n_estimators': 100,
        'max_depth': 5,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'max_features': 'sqrt',
        'random_state': 42
    }
    
    PRICE_MODEL_PARAMS = {
        'n_estimators': 100,
        'max_depth': 5,
        'learning_rate': 0.1,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'random_state': 42
    }
    
    # 交易參數
    COMMISSION_RATE = 0.001425  # 手續費率
    TRADE_TAX_RATE = 0.003     # 交易稅率
    MIN_PROFIT_THRESHOLD = 0.02  # 最小獲利閾值

class ModelEvaluator:
    """模型評估器"""
    @staticmethod
    def calculate_trading_metrics(y_true: np.ndarray, y_pred: np.ndarray, 
                                prices: np.ndarray) -> Dict[str, float]:
        """計算交易相關指標"""
        # 模擬交易結果
        positions = np.where(y_pred == 1, 1, -1)  # 1代表做多,-1代表做空
        returns = np.diff(prices) / prices[:-1]
        strategy_returns = positions[:-1] * returns
        
        # 計算交易成本
        trades = np.diff(positions) != 0
        total_trades = np.sum(trades)
        transaction_costs = (ModelConfig.COMMISSION_RATE * 2 + ModelConfig.TRADE_TAX_RATE) * total_trades
        
        # 計算績效指標
        cumulative_returns = np.cumprod(1 + strategy_returns) - 1
        annualized_return = (1 + cumulative_returns[-1]) ** (252/len(returns)) - 1
        sharpe_ratio = np.sqrt(252) * np.mean(strategy_returns) / np.std(strategy_returns)
        max_drawdown = np.min(cumulative_returns - np.maximum.accumulate(cumulative_returns))
        
        return {
            'annualized_return': annualized_return,
            'sharpe_ratio': sharpe_ratio,
            'max_drawdown': max_drawdown,
            'total_trades': total_trades,
            'transaction_costs': transaction_costs,
            'final_return': cumulative_returns[-1]
        }
    
    @staticmethod
    def calculate_prediction_metrics(y_true: np.ndarray, y_pred: np.ndarray, 
                                  y_prob: np.ndarray = None) -> Dict[str, float]:
        """計算預測相關指標"""
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'confusion_matrix': confusion_matrix(y_true, y_pred)
        }
        
        if y_prob is not None:
            metrics['roc_auc'] = roc_auc_score(y_true, y_prob[:, 1])
        
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='binary'
        )
        
        metrics.update({
            'precision': precision,
            'recall': recall,
            'f1': f1
        })
        
        return metrics

In [59]:
class EnhancedFeatureGenerator:
    """Enhanced feature generator for stock prediction"""
    
    def __init__(self):
        self.feature_names = None
    
    def prepare_training_data(self, stock_data: pd.DataFrame) -> Tuple[np.ndarray, List[str]]:
        """Prepare training data with features"""
        # Define feature names
        self.feature_names = [
            '趨勢強度', '均線糾結度', '波動率', '振幅', '量能趨勢',
            '量比', 'RSI_動能', 'MACD_動能', 'RSI', 'MACD',
            'KD_差值', '本益比_相對值', '技術綜合評分'
        ]
        
        # Calculate features (using the data already processed in your code)
        features = stock_data[self.feature_names].values
        
        return features, self.feature_names

class StockPredictor:
    """改進的股票預測模型"""
    
    def __init__(self):
        self.feature_generator = EnhancedFeatureGenerator()
        self.trend_model = RandomForestClassifier(**ModelConfig.TREND_MODEL_PARAMS)
        self.price_model = GradientBoostingRegressor(**ModelConfig.PRICE_MODEL_PARAMS)
        self.analyzer = ModelAnalyzer()
        self.evaluator = ModelEvaluator()
        
        # 設定模型優化參數
        self.trend_model_params = {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 7],
            'min_samples_split': [2, 5, 10]
        }
        self.optimizer = ModelOptimizer(
            self.trend_model, 
            self.trend_model_params
        )

    def train_and_evaluate(self, stock_data: pd.DataFrame, optimize: bool = True) -> Dict[str, Any]:
        """訓練和評估模型"""
        try:
            logging.info("開始訓練和評估模型...")
            
            # 準備數據
            features, trend_labels, price_changes, feature_names = self.prepare_data(stock_data)
            
            # 優化模型參數(如果需要)
            optimization_results = None
            if optimize:
                logging.info("開始優化模型參數...")
                optimization_results = self.optimizer.optimize(
                    features, trend_labels
                )
                self.trend_model.set_params(**optimization_results['best_params'])
                logging.info(f"最佳參數: {optimization_results['best_params']}")
            
            # 使用時間序列交叉驗證
            results = self._perform_time_series_cv(
                features, trend_labels, price_changes, stock_data
            )
            
            # 添加優化結果
            if optimization_results:
                results['optimization_results'] = optimization_results
            
            # 視覺化分析
            self._visualize_results(results, stock_data)
            
            return results
            
        except Exception as e:
            logging.error(f"模型訓練和評估過程出錯: {str(e)}")
            raise

    def prepare_data(self, stock_data: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str]]:
        """準備訓練數據"""
        try:
            # 生成特徵
            features, feature_names = self.feature_generator.prepare_training_data(stock_data)
            features = features[:-1]  # 移除最後一行
            
            # 準備標籤
            trend_labels = np.where(stock_data['收盤價'].shift(-1) > stock_data['收盤價'], 1, 0)[:-1]
            price_changes = stock_data['收盤價'].pct_change().shift(-1).values[:-1]
            
            return features, trend_labels, price_changes, feature_names
            
        except KeyError as e:
            logging.error(f"數據中缺少必要的列: {str(e)}")
            raise
        except Exception as e:
            logging.error(f"準備數據時出錯: {str(e)}")
            raise

    def _perform_time_series_cv(self, features: np.ndarray, 
                              trend_labels: np.ndarray,
                              price_changes: np.ndarray,
                              stock_data: pd.DataFrame) -> Dict[str, Any]:
        """執行時間序列交叉驗證"""
        tscv = TimeSeriesSplit(n_splits=5)
        cv_results = []
        
        for train_idx, test_idx in tscv.split(features):
            fold_result = self._evaluate_fold(
                features, trend_labels, price_changes, 
                stock_data, train_idx, test_idx
            )
            cv_results.append(fold_result)
        
        return self._aggregate_cv_results(cv_results)

    def _evaluate_fold(self, features: np.ndarray, 
                      trend_labels: np.ndarray,
                      price_changes: np.ndarray,
                      stock_data: pd.DataFrame,
                      train_idx: np.ndarray,
                      test_idx: np.ndarray) -> Dict[str, Any]:
        """評估單個交叉驗證折"""
        # 分割數據
        X_train = features[train_idx]
        X_test = features[test_idx]
        y_trend_train = trend_labels[train_idx]
        y_trend_test = trend_labels[test_idx]
        y_price_train = price_changes[train_idx]
        
        # 訓練模型
        self.trend_model.fit(X_train, y_trend_train)
        self.price_model.fit(X_train, y_price_train)
        
        # 預測
        trend_pred = self.trend_model.predict(X_test)
        trend_prob = self.trend_model.predict_proba(X_test)
        price_pred = self.price_model.predict(X_test)
        
        # 計算各種指標
        prediction_metrics = self.evaluator.calculate_prediction_metrics(
            y_trend_test, trend_pred, trend_prob
        )
        
        # 生成交易信號
        signals = np.where(trend_prob[:, 1] > 0.5, 1, -1)
        
        trading_metrics = self.evaluator.calculate_trading_metrics(
            y_trend_test, trend_pred, 
            stock_data['收盤價'].values[test_idx]
        )
        
        return {
            'prediction_metrics': prediction_metrics,
            'trading_metrics': trading_metrics,
            'predictions': {
                'trend': trend_pred,
                'probability': trend_prob,
                'price': price_pred,
                'signals': signals
            },
            'test_indices': test_idx
        }

    def _aggregate_cv_results(self, cv_results: List[Dict[str, Any]]) -> Dict[str, Any]:
        """聚合交叉驗證結果"""
        # 合併預測結果
        all_predictions = {
            'y_true': np.concatenate([r['predictions']['trend'] for r in cv_results]),
            'y_pred': np.concatenate([r['predictions']['trend'] for r in cv_results]),
            'probabilities': np.concatenate([r['predictions']['probability'] for r in cv_results]),
            'signals': np.concatenate([r['predictions']['signals'] for r in cv_results])
        }
        
        # 計算平均指標
        prediction_metrics = {}
        trading_metrics = {}
        
        for metric in cv_results[0]['prediction_metrics']:
            values = [r['prediction_metrics'][metric] for r in cv_results]
            prediction_metrics[metric] = np.mean(values)
        
        for metric in cv_results[0]['trading_metrics']:
            values = [r['trading_metrics'][metric] for r in cv_results]
            trading_metrics[metric] = np.mean(values)
        
        # 特徵重要性
        feature_importance = dict(zip(
            self.feature_generator.feature_names,
            self.trend_model.feature_importances_
        ))
        
        return {
            'evaluation': {
                'prediction': prediction_metrics,
                'trading': trading_metrics,
                **all_predictions
            },
            'feature_importance': feature_importance,
            'cv_results': cv_results
        }

    def _visualize_results(self, results: Dict[str, Any], 
                          stock_data: pd.DataFrame) -> None:
        """視覺化分析結果"""
        try:
            # 繪製累積報酬曲線
            if 'trading' in results['evaluation']:
                returns = pd.Series(
                    results['evaluation']['signals'],
                    index=stock_data.index[:-1]
                )
                self.analyzer.plot_cumulative_returns(returns)
            
            # 繪製混淆矩陣
            if 'y_true' in results['evaluation'] and 'y_pred' in results['evaluation']:
                self.analyzer.plot_confusion_matrix(
                    results['evaluation']['y_true'],
                    results['evaluation']['y_pred']
                )
            
            # 繪製ROC曲線
            if 'probabilities' in results['evaluation']:
                self.analyzer.plot_roc_curve(
                    results['evaluation']['y_true'],
                    results['evaluation']['probabilities']
                )
            
            # 繪製特徵重要性
            if 'feature_importance' in results:
                self.analyzer.plot_feature_importance(
                    results['feature_importance']
                )
            
        except Exception as e:
            logging.error(f"視覺化結果時出錯: {str(e)}")
            raise

In [60]:
def main():
    """主程式"""
    try:
        # 設置日誌
        setup_logging()
        
        # 載入測試數據
        stock_id = '2330'
        start_date = '2023-01-01'
        end_date = '2024-01-01'
        
        logging.info(f"載入股票 {stock_id} 的數據...")
        stock_data = load_stock_data(stock_id, start_date, end_date)
        
        # 初始化預測器和分析器
        predictor = StockPredictor()
        analyzer = ModelAnalyzer()
        
        # 訓練和評估模型(包含參數優化)
        logging.info("開始模型訓練和評估...")
        results = predictor.train_and_evaluate(stock_data, optimize=True)
        
        # 輸出模型評估結果
        logging.info("\n=== 模型評估結果 ===")
        
        # 預測效能指標
        logging.info("\n預測指標:")
        for metric, value in results['evaluation']['prediction'].items():
            logging.info(f"{metric}: {value:.4f}")
        
        # 交易績效指標
        logging.info("\n交易指標:")
        for metric, value in results['evaluation']['trading'].items():
            logging.info(f"{metric}: {value:.4f}")
        
        # 視覺化分析
        logging.info("\n生成視覺化分析...")
        
        # 1. 繪製績效指標雷達圖
        performance_metrics = {
            '準確率': results['evaluation']['prediction']['accuracy'],
            'ROC AUC': results['evaluation']['prediction']['roc_auc'],
            '年化報酬': results['evaluation']['trading']['annualized_return'],
            '夏普比率': results['evaluation']['trading']['sharpe_ratio'],
            '最大回撤': -results['evaluation']['trading']['max_drawdown']
        }
        analyzer.plot_performance_metrics(performance_metrics, 
                                       title=f"{stock_id} 模型績效指標")
        
        # 2. 繪製累積報酬曲線
        returns = pd.Series(
            results['trading_metrics']['strategy_returns'],
            index=stock_data.index[:-1]
        )
        analyzer.plot_cumulative_returns(
            returns,
            title=f"{stock_id} 策略累積報酬"
        )
        
        # 3. 繪製混淆矩陣
        analyzer.plot_confusion_matrix(
            results['evaluation']['y_true'],
            results['evaluation']['y_pred'],
            title=f"{stock_id} 預測混淆矩陣"
        )
        
        # 4. 繪製ROC曲線
        analyzer.plot_roc_curve(
            results['evaluation']['y_true'],
            results['evaluation']['probabilities'],
            title=f"{stock_id} ROC曲線"
        )
        
        # 5. 繪製特徵重要性
        analyzer.plot_feature_importance(
            results['feature_importance'],
            title=f"{stock_id} 特徵重要性排序"
        )
        
        # 6. 繪製交易信號
        analyzer.plot_trading_signals(
            stock_data['收盤價'],
            results['trading_metrics']['signals'],
            title=f"{stock_id} 交易信號"
        )
        
        # 輸出優化結果
        if 'optimization_results' in results:
            logging.info("\n=== 參數優化結果 ===")
            logging.info(f"最佳參數: {results['optimization_results']['best_params']}")
            logging.info(f"最佳分數: {results['optimization_results']['best_score']:.4f}")
        
        logging.info("\n分析完成!")
        
    except Exception as e:
        logging.error(f"執行過程發生錯誤: {str(e)}")
        raise

if __name__ == "__main__":
    main()

2024-10-29 18:44:19,216 - INFO - 載入股票 2330 的數據...
2024-10-29 18:44:19,217 - INFO - 正在讀取數據文件: D:\Min\Python\Project\FA_Data\meta_data\all_stocks_data.csv
2024-10-29 18:44:28,908 - INFO - 原始數據形狀: (2450900, 29)
2024-10-29 18:44:29,109 - INFO - 過濾後的數據形狀 (股票 2330): (2584, 29)
2024-10-29 18:44:29,112 - INFO - 開始日期過濾後的數據形狀: (437, 29)
2024-10-29 18:44:29,114 - INFO - 結束日期過濾後的數據形狀: (239, 29)
2024-10-29 18:44:29,125 - INFO - 開始模型訓練和評估...
2024-10-29 18:44:29,127 - INFO - 開始訓練和評估模型...
2024-10-29 18:44:29,127 - ERROR - 數據中缺少必要的列: "['趨勢強度', '均線糾結度', '波動率', '振幅', '量能趨勢', '量比', 'RSI_動能', 'MACD_動能', 'KD_差值', '本益比_相對值', '技術綜合評分'] not in index"
2024-10-29 18:44:29,128 - ERROR - 模型訓練和評估過程出錯: "['趨勢強度', '均線糾結度', '波動率', '振幅', '量能趨勢', '量比', 'RSI_動能', 'MACD_動能', 'KD_差值', '本益比_相對值', '技術綜合評分'] not in index"
2024-10-29 18:44:29,128 - ERROR - 執行過程發生錯誤: "['趨勢強度', '均線糾結度', '波動率', '振幅', '量能趨勢', '量比', 'RSI_動能', 'MACD_動能', 'KD_差值', '本益比_相對值', '技術綜合評分'] not in index"



=== 股票數據摘要 ===
股票代碼: 2330
股票名稱: 台積電
數據期間: 2023-01-03 到 2023-12-29
交易天數: 239 天

--- 價格統計 ---
平均收盤價: 543.45
最高價: 594.00
最低價: 443.00
價格波動範圍: 151.00
標準差: 29.62

--- 交易統計 ---
平均日成交量: 27,535,566.40
平均日成交筆數: 29,635.65
平均日成交金額: 14,980,440,520.00

--- 技術指標概覽 ---

TREND類指標:
SMA30        - 均值: 536.47, 標準差: 28.85
DEMA30       - 均值: 543.00, 標準差: 29.66
EMA30        - 均值: 536.41, 標準差: 28.16
TSF          - 均值: 543.78, 標準差: 31.83
middleband   - 均值: 566.27, 標準差: 26.47

MOMENTUM類指標:
RSI          - 均值: 53.59, 標準差: 10.37
MACD         - 均值: 3.21, 標準差: 7.07
MACD_signal  - 均值: 3.12, 標準差: 6.46
MACD_hist    - 均值: 0.09, 標準差: 2.59

VOLATILITY類指標:
slowk        - 均值: 52.71, 標準差: 28.52
slowd        - 均值: 52.39, 標準差: 25.99
SAR          - 均值: 539.43, 標準差: 34.31


KeyError: "['趨勢強度', '均線糾結度', '波動率', '振幅', '量能趨勢', '量比', 'RSI_動能', 'MACD_動能', 'KD_差值', '本益比_相對值', '技術綜合評分'] not in index"